# AMO Case — Продуктовый аналитик

**Контекст:** пол года назад подписку подняли с 6.99 до 9.99/нед (тест показал, что конверсии не упали). CEO сомневается: цена выросла на 42%, а в реальном Revenue таких цифр нет. Наша задача — разобраться.

## Дорожная карта

| Блок | Что делаем |
|---|---|
| **0** | Загрузка + чистка (дубли, типы, sanity-check) |
| **1** | A/B: конверсии click → purchase → sub_purchase по сегментам + стат.значимость |
| **2** | LTV на 1.5 мес для 6.99 vs 9.99 (с учётом refund / chargeback / OTP) |
| **3** | Реально ли ARPU вырос на +42%? Деньги за период 26.01–16.03 |
| **4** | ROAS по сегментам (канал / регион / ценовая группа) |
| **Итог** | Вывод + рекомендация по цене |

---
## Блок 0. Настройка окружения и загрузка данных

### 🎯 Твоя задача (пиши код сам в ячейке ниже):

1. **Импортировать библиотеки** (стандартные псевдонимы):
   - `pandas` → `pd`
   - `numpy` → `np`
   - `matplotlib.pyplot` → `plt`
   - `seaborn` → `sns`
2. (по желанию) настроить отображение, чтобы видеть все колонки: `pd.set_option('display.max_columns', None)`
3. **Загрузить 6 CSV** из папки `data/` через `pd.read_csv(...)` в 6 отдельных переменных (датафреймов):
   - `test_data.csv`, `order_data.csv`, `otp_data.csv`, `refunds_cb_data.csv`, `user_data.csv`, `marketing_spend_data.csv`

**Подсказки по путям:** ноутбук лежит в корне проекта, данные — в подпапке `data`. Значит относительный путь = `'data/test_data.csv'` и т.д.

**Что вспомнить:** `pd.read_csv('путь')` возвращает датафрейм. Присвой его переменной: `df = pd.read_csv(...)`.

In [6]:
# --- 1. Импорты (напиши сам) ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 2. Настройки отображения (по желанию) ---
pd.set_option('display.max_columns', None)

# --- 3. Загрузка 6 CSV из папки data/ (напиши сам) ---
df_order_data = pd.read_csv('data/order_data.csv')
df_otp_data = pd.read_csv('data/otp_data.csv')
df_refunds_data = pd.read_csv('data/refunds_cb_data.csv')
df_test_data = pd.read_csv('data/test_data.csv')
df_user_data = pd.read_csv('data/user_data.csv')
df_marketing_spend_data = pd.read_csv('data/marketing_spend_data.csv')



### Быстрая проверка, что всё загрузилось

После загрузки выведи размер каждой таблицы (`.shape`) и посмотри на первые строки (`.head()`). Начни с одной таблицы — например `test_data.head()`.

In [12]:
# Проверка загрузки (напиши сам)
print(f"Данные о маркетинговых расходах: {df_marketing_spend_data.shape[0]} строк, {df_marketing_spend_data.shape[1]} столбцов")
print(f"Данные о заказах: {df_order_data.shape[0]} строк, {df_order_data.shape[1]} столбцов")
print(f"Данные об OTP: {df_otp_data.shape[0]} строк, {df_otp_data.shape[1]} столбцов")
print(f"Данные о возвратах: {df_refunds_data.shape[0]} строк, {df_refunds_data.shape[1]} столбцов")
print(f"Данные о тестах: {df_test_data.shape[0]} строк, {df_test_data.shape[1]} столбцов")
print(f"Данные о пользователях: {df_user_data.shape[0]} строк, {df_user_data.shape[1]} столбцов")



Данные о маркетинговых расходах: 609 строк, 7 столбцов
Данные о заказах: 342616 строк, 6 столбцов
Данные об OTP: 44110 строк, 3 столбцов
Данные о возвратах: 215088 строк, 5 столбцов
Данные о тестах: 127598 строк, 9 столбцов
Данные о пользователях: 215088 строк, 5 столбцов


In [23]:
display(df_marketing_spend_data.head())
display(df_order_data.head())
display(df_otp_data.head())
display(df_refunds_data.head())
display(df_test_data.head())
display(df_user_data.head())


,date,channel,country_group,group,clicks,cpc,spend
0,2025-01-05,Facebook,EU,control,2,0.6992,1.40
1,2025-01-05,Facebook,EU,test,1,0.7836,0.78
2,2025-01-05,Facebook,Tier1,test,1,0.7426,0.74
3,2025-01-05,Facebook,USA,control,4,0.7768,3.11
4,2025-01-05,Facebook,USA,test,2,0.8074,1.61


,user_uuid,date_started,Product Name,order_date,Trial Period Days,Rebill Period Days
0,9a15cd24-d957-49c6-9e83-1f5cb8897280,2025-02-19,weekly 6.99,2025-02-26,7,7
1,9a15cd24-d957-49c6-9e83-1f5cb8897280,2025-02-19,weekly 6.99,2025-03-05,7,7
2,f5fa6241-e57f-45d9-9013-ff0956fb5d91,2025-03-14,weekly 6.99,2025-03-21,7,7
3,6face9f9-76dd-44fb-bfc8-d5676071b311,2025-02-26,weekly 6.99,2025-03-05,7,7
4,30599dfe-abda-442d-9ba4-9a1c7cf9ec17,2025-02-12,weekly 9.99,2025-02-19,7,7


,user_uuid,Product Name,revenue
0,dfd3ab19-6189-4404-82e4-f1f0c0a3922c,one time payment,15
1,0b62b33e-b439-445e-9758-6f604c865609,one time payment,15
2,7802c61c-5a0d-4942-aca6-1f3908474add,one time payment,15
3,3d4ceb0e-b061-48d9-a717-4fa32e737053,one time payment,15
4,4bb69bca-52c6-4d99-be62-328cdd4430d4,one time payment,15


,user_uuid,Product Name,refund,chargeback,price
0,9a15cd24-d957-49c6-9e83-1f5cb8897280,weekly 6.99,0,0,6.99
1,f5fa6241-e57f-45d9-9013-ff0956fb5d91,weekly 6.99,0,0,6.99
2,6face9f9-76dd-44fb-bfc8-d5676071b311,weekly 6.99,0,0,6.99
3,30599dfe-abda-442d-9ba4-9a1c7cf9ec17,weekly 9.99,0,0,9.99
4,b6c932a7-174e-4ff3-8e95-17342b4e54c9,weekly 6.99,0,0,6.99


,user_id,event_date,event_name,group,country_group,platform,payment_method,product_name,channel
0,bd02dd19-1250-4902-877a-818617262946,2025-01-18,click,control,Tier1,Android,card,Weekly 6.99,Facebook
1,4c78ff37-7f18-412f-aff3-b7109100b922,2025-01-19,click,control,EU,Android,card,Weekly 6.99,Organic
2,9e2ced6a-dd70-4836-ba96-ee71c4165980,2025-01-13,click,test,USA,IOS,paypal,Weekly 9.99,Google
3,bab3f720-3ada-47a8-b9c3-e9310bd4ef7c,2025-01-13,click,test,EU,Android,paypal,Weekly 9.99,Google
4,eca01a89-0126-4d33-ac46-a0481dcd7220,2025-01-24,purchase,test,USA,IOS,card,Weekly 9.99,Facebook


,user_uuid,region,platform,payment_type,age_segment
0,9a15cd24-d957-49c6-9e83-1f5cb8897280,USA,Android,card,31-45
1,f5fa6241-e57f-45d9-9013-ff0956fb5d91,USA,IOS,card,31-45
2,6face9f9-76dd-44fb-bfc8-d5676071b311,USA,Android,card,31-45
3,30599dfe-abda-442d-9ba4-9a1c7cf9ec17,USA,IOS,card,18-30
4,b6c932a7-174e-4ff3-8e95-17342b4e54c9,USA,Android,card,31-45


## Блок 0.1: качество данных

In [24]:
dfs = {'test': df_test_data, 'order': df_order_data, 'otp': df_otp_data,
       'refunds': df_refunds_data, 'user': df_user_data, 'mkt': df_marketing_spend_data}

for name, df in dfs.items():
    print(name, '→ полных дублей:', df.duplicated().sum())

df_test_data[df_test_data.duplicated(keep=False)]['group'].value_counts()

test → полных дублей: 1131
order → полных дублей: 0
otp → полных дублей: 0
refunds → полных дублей: 0
user → полных дублей: 0
mkt → полных дублей: 0


group
control    1135
Name: count, dtype: int64